In [10]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from multiprocessing import Pool


In [11]:
LINES = [
    (0, 1, 2),
    (3, 4, 5),
    (6, 7, 8),
    (0, 3, 6),
    (1, 4, 7),
    (2, 5, 8),
    (0, 4, 8),
    (2, 4, 6)
]

def moves(s):
    return [i for i in range(9) if s[i] == 0]


def play(s, a):
    s = s.copy()
    s[a] = 1
    return -s


def result(s):
    """Value for the player to move, or None if game continues."""
    
    for a, b, c in LINES:
        if s[a] == s[b] == s[c] == -1:
            return -1.0
    
    return 0.0 if 0 not in s else None

In [12]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.body = nn.Sequential(
            nn.Linear(9, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
        
        self.pi = nn.Linear(64, 9)
        self.v = nn.Linear(64, 1)

    def forward(self, x):
        h = self.body(x)
        return self.pi(h), torch.tanh(self.v(h)).squeeze(-1)

In [13]:
def predict(net, s):
    with torch.no_grad():
        x = torch.tensor(s).float().unsqueeze(0)
        logits, v = net(x)
        
        p = F.softmax(logits, 1)[0].numpy()
    
    # Allow only legal moves
    mask = np.zeros(9)
    mask[moves(s)] = 1
    
    p = p * mask
    
    return p / p.sum(), v.item()

In [14]:
class Node:
    def __init__(self, p):
        self.p = p
        self.n = 0
        self.w = 0.0
        self.kids = {}

In [15]:
def expand(net, node, s):
    p, v = predict(net, s)
    
    for a in moves(s):
        node.kids[a] = Node(p[a])
    
    return v

In [16]:
def mcts(net, s0, sims=50, c=1.5):
    root = Node(1)
    expand(net, root, s0)

    for _ in range(sims):
        node = root
        s = s0
        path = [root]

        # Selection
        while node.kids:
            N = math.sqrt(node.n + 1)

            a, node = max(
                node.kids.items(),
                key=lambda kv:
                    (
                        (-kv[1].w / kv[1].n if kv[1].n else 0)
                        + c * kv[1].p * N / (1 + kv[1].n)
                    )
            )

            s = play(s, a)
            path.append(node)

        # Evaluation
        v = result(s)

        if v is None:
            v = expand(net, node, s)

        # Backup
        for n in reversed(path):
            n.n += 1
            n.w += v
            v = -v

    # Improved policy from visit counts
    pi = np.zeros(9)

    for a, k in root.kids.items():
        pi[a] = k.n

    return pi / pi.sum()

In [17]:
def self_play(net, sims=50):
    s = np.zeros(9, dtype=np.float32)
    hist = []

    while True:
        pi = mcts(net, s, sims)
        hist.append((s.copy(), pi))

        action = np.random.choice(9, p=pi)
        s = play(s, action)

        r = result(s)

        if r is not None:
            break

    # Prepare training examples
    return [
        (st, p, -r if i % 2 == 0 else r)
        for i, (st, p) in enumerate(reversed(hist))
    ]

In [18]:
def worker(args):
    state_dict, n_games = args
    
    torch.set_num_threads(1)
    
    net = Net()
    net.load_state_dict(state_dict)

    return [
        ex
        for _ in range(n_games)
        for ex in self_play(net)
    ]

In [19]:
def train(net, data, steps=50):
    opt = torch.optim.Adam(net.parameters(), 1e-3)

    S = torch.tensor(
        np.array([d[0] for d in data])
    ).float()

    P = torch.tensor(
        np.array([d[1] for d in data])
    ).float()

    Z = torch.tensor(
        [d[2] for d in data]
    ).float()

    for _ in range(steps):
        logits, v = net(S)

        policy_loss = -(
            P * F.log_softmax(logits, 1)
        ).sum(1).mean()

        value_loss = F.mse_loss(v, Z)

        loss = policy_loss + value_loss

        opt.zero_grad()
        loss.backward()
        opt.step()

    return loss.item()

In [20]:
net = Net()
buffer = []

for it in range(10):
    
    games = []
    
    for _ in range(8):
        games.extend(self_play(net, sims=50))
    
    buffer += games
    
    # Keep only recent examples
    buffer = buffer[-5000:]
    
    loss = train(net, buffer)
    
    print(
        f"iter {it + 1}: "
        f"examples={len(buffer)}, "
        f"loss={loss:.3f}"
    )

iter 1: examples=53, loss=2.560
iter 2: examples=106, loss=2.313
iter 3: examples=165, loss=2.182
iter 4: examples=221, loss=2.106
iter 5: examples=277, loss=1.976
iter 6: examples=328, loss=1.863
iter 7: examples=385, loss=1.784
iter 8: examples=438, loss=1.706
iter 9: examples=493, loss=1.619
iter 10: examples=551, loss=1.565


In [21]:
s = np.zeros(9, dtype=np.float32)

print("Initial board:")
print(s.reshape(3, 3))

pi = mcts(net, s, sims=100)

print("\nMove probabilities:")
print(pi.reshape(3, 3))

print("\nBest move:", np.argmax(pi))

Initial board:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

Move probabilities:
[[0.   0.   0.33]
 [0.   0.59 0.  ]
 [0.02 0.   0.06]]

Best move: 4
